John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

Adapted from David Logue's code

### Import Packages

In [1]:
# Import packages

library(geosphere) # For distm(), for calculating distances
library(tidyverse)
# library(dagitty)
# library(brms)
# library(forestplot)
# library(lqmm)

Warning message:
"package 'geosphere' was built under R version 4.4.3"
Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'tibble' was built under R version 4.4.3"
Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'readr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'forcats' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filt

### Load and Trim Dataset

In [2]:
data <- read_csv("../data/megazord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

New names:
• `` -> `...1`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 12799 Columns: 127
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (14): ID, recordist, Begin.Path, Location_Number, Comments, ST1, Data_...
dbl  (104): ...1, X.2, X.1, selec, start, end, File.Offset..s., SNR.NIST.Qui...
lgl    (2): RepeatST, GreaterThanT
dttm   (1): DateTime
date   (1): date
time   (5): click_clock, start_hms, click_clock_hms, clock_time, sunrise

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [3]:
# Get all unique focal male IDs (excluding NAs)
focal_ids <- unique(na.omit(data$ID))
print(focal_ids)

# Initialize an empty list to store the output dataframes
focal_outputs <- list()

# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
# Get rows in which ID == focal_id
focals_data <- filter(data, ID == focal_id)

# Get unique set of USTs for the focal (including NA values)
focal_repertoire <- unique(focals_data$UST)

# Find all rows that have a UST in focal's repertoire, not including NA values
focal_ind <- data %>% filter(UST %in% focal_repertoire & !is.na(UST))
# Group these rows by date, create columns to show focal's next UST
focal_ind <- focal_ind %>%
    group_by(date) %>% # Group by date so we're not comparing "last song" across days
    mutate(focal_next_UST = if_else(ID == focal_id, UST, NA)) %>%
    fill(focal_next_UST, .direction = "up")

# Create column focal_matches : binary variable (is there a song type match?)
# If the focal bird is the singer, it cannot match itself, return 0
# If the focal bird is not the singer, does the song type match the focal's next song? (within day)
focal_ind <- focal_ind %>%
    mutate(focal_matches = if_else(ID == focal_id, NA,
                                    if_else(focal_next_UST == UST, 1, 0)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


### Calculate Distances for Each Matching Opportunity

In [4]:
# Coordinates for focal male and singing male
coords1 <- cbind(focal_ind$M1_Lat, focal_ind$M1_Lon) #TODO: Check if M1 is the correct male here
coords2 <- cbind(focal_ind$Lat, focal_ind$Lon)

# Distance matrix is a function of the two coordinate sets
dist_matrix <- distm(coords1, coords2, fun = distHaversine)
# focal_ind %>% mutate(dist_to_focal = diag(dist_matrix))
focal_ind$dist_to_focal <- diag(dist_matrix)


### Create other Columns

In [5]:
# Make column "Focal_ID"
focal_ind$Focal_ID <- focal_id

In [6]:
# Make column for time since focal male's last song
# NA values at top of each day until focal male sings
focal_ind <- focal_ind %>% 
    group_by(date) %>%
    mutate(time_focals_last_song = if_else(ID == focal_id, time_rel_sunrise, NA)) %>%
    fill(time_focals_last_song, .direction = "down") %>%
    mutate(time_since_focals_last_song = time_rel_sunrise - time_focals_last_song) %>%
    ungroup() %>%
    # make column which contains the time of focal's songs (like "time_focal's_last_song", but not f-filled)
    mutate(just_focal_time = ifelse(ID == focal_id, time_rel_sunrise, NA))

In [7]:
# Group by date and UST to make column showing time focal last sang a song type
focal_ind <- focal_ind %>%
    group_by(date, UST) %>%
    mutate(time_focals_last_rend_ST = lag(just_focal_time)) %>%
    fill(time_focals_last_rend_ST, .direction = "down") %>%
    mutate(time_since_focals_last_rend_ST = -1 * (time_focals_last_rend_ST - time_rel_sunrise))

In [8]:
# Select only necessary variables
focal_ind_lean <- focal_ind %>%
    select(index, Focal_ID, date, ID, time_rel_sunrise, UST, number_of_matches,
        does_it_match, focal_matches, dist_to_focal, time_since_focals_last_song,
        time_focals_last_rend_ST, time_since_focals_last_rend_ST, focal_next_UST:time_since_focals_last_rend_ST) %>%
# Remove focal's songs (which he cannot match)
    filter(ID != focal_id) %>%
    ungroup()

In [9]:
head(focal_ind_lean)

index,Focal_ID,date,ID,time_rel_sunrise,UST,number_of_matches,does_it_match,focal_matches,dist_to_focal,time_since_focals_last_song,time_focals_last_rend_ST,time_since_focals_last_rend_ST,focal_next_UST,time_focals_last_song,just_focal_time
<dbl>,<chr>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3,LbOW,2022-05-13,DgTbO,-1909.821,20,0,0,0,NA,NA,NA,NA,10,NA,NA
7,LbOW,2022-05-13,DgTbO,-1871.335,8,1,1,0,NA,NA,NA,NA,10,NA,NA
9,LbOW,2022-05-13,DgTbO,-1854.549,20,0,0,0,NA,NA,NA,NA,10,NA,NA
11,LbOW,2022-05-13,DgTbO,-1842.433,10,0,0,1,NA,NA,NA,NA,10,NA,NA
19,LbOW,2022-05-13,DgTbO,-1770.057,20,0,0,0,NA,NA,NA,NA,10,NA,NA
21,LbOW,2022-05-13,DgTbO,-1755.132,8,0,0,0,NA,NA,NA,NA,10,NA,NA


### Export Data

In [10]:
write_csv(focal_ind_lean, "../data/all_matching_opportunities.csv")